<a href="https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Apache%20Spark/3-DataFrames_and_Spark_SQL.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<h1 style="text-align:center; color:#E25822; font-size:2.4em; font-family:Arial,sans-serif;">
  DataFrames and Spark SQL in PySpark
</h1>
<h3 style="text-align:center; color:#555; font-family:Arial,sans-serif;">
  Big Data Engineering &mdash; Module 5
</h3>
<hr style="border:2px solid #E25822;">

This notebook provides a comprehensive hands-on introduction to **PySpark DataFrames** and **Spark SQL** — the two primary high-level APIs for structured data processing in Apache Spark.

### What you will learn
| Topic | Description |
|---|---|
| DataFrames vs RDDs | When to use each API |
| Creating DataFrames | Multiple creation methods |
| DataFrame API | select, filter, withColumn, join, and more |
| Aggregations | groupBy, agg, count, avg, sum |
| Built-in Functions | String, numeric, date, and conditional functions |
| Missing Data | dropna, fillna, null counting |
| Spark SQL | SQL queries on DataFrames |
| Reading & Writing | CSV, JSON, Parquet with partitioning |

> **Dataset:** Throughout this notebook we use an **apartment prices** dataset (`apartment_prices.csv`) with columns: `City`, `Bedrooms`, `Bathrooms`, `Square_Area`, `Price`.

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  1. Environment Setup
</h2>

Install PySpark and create a **SparkSession** — the single entry point to all Spark functionality.

In [40]:
# Install PySpark (required on Google Colab; skip if running locally)
!pip install pyspark --quiet

In [41]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFrames and Spark SQL") \
    .master("local[*]") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

# Reduce verbose logging
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"Python version: ", end="")
import sys; print(sys.version.split()[0])

Spark version : 4.0.2
Python version: 3.12.13


**Expected output:**
```
Spark version : 3.x.x
Python version: 3.x.x
```

> `local[*]` tells Spark to run locally using all available CPU cores. In production the master URL points to a cluster manager (YARN, Kubernetes, etc.).

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  2. DataFrames vs RDDs
</h2>

Spark provides two main distributed data abstractions. Understanding when to use each is important.

| Feature | RDD | DataFrame |
|---|---|---|
| Introduced | Spark 1.0 | Spark 1.3 |
| Abstraction level | Low-level | High-level (structured) |
| Schema | No — untyped | Yes — named columns + types |
| Optimization engine | None (manual) | Catalyst + Tungsten (automatic) |
| Performance | Slower (no optimization) | Faster (optimized execution plan) |
| API style | Functional (map, filter, reduce) | SQL-like (select, filter, groupBy) |
| Language support | Scala, Java, Python, R | Scala, Java, Python, R |
| Best for | Unstructured data, custom logic | Structured/semi-structured data |
| Interoperability | Can convert to/from DataFrame | Can convert to/from RDD |

### Key takeaway
- Use **DataFrames** (or Spark SQL) for structured data — they are faster and easier to write.
- Use **RDDs** only when you need fine-grained control or are working with unstructured data.

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  3. Creating DataFrames
</h2>

PySpark DataFrames can be created from many sources. We cover the four most common approaches.

### 3.1 From a List of `Row` Objects

The most explicit method — each `Row` defines one record. The schema is inferred automatically from the field names and Python types.

In [42]:
from pyspark.sql import Row

data = [
    Row(City="Amman",    Bedrooms=3, Bathrooms=2, Square_Area=150.0, Price=120000.0),
    Row(City="Zarqa",    Bedrooms=2, Bathrooms=1, Square_Area=90.0,  Price=70000.0),
    Row(City="Irbid",    Bedrooms=4, Bathrooms=3, Square_Area=200.0, Price=160000.0),
    Row(City="Amman",    Bedrooms=1, Bathrooms=1, Square_Area=60.0,  Price=50000.0),
    Row(City="Aqaba",    Bedrooms=3, Bathrooms=2, Square_Area=140.0, Price=115000.0),
    Row(City="Amman",    Bedrooms=5, Bathrooms=4, Square_Area=300.0, Price=250000.0),
    Row(City="Zarqa",    Bedrooms=3, Bathrooms=2, Square_Area=130.0, Price=95000.0),
    Row(City="Irbid",    Bedrooms=2, Bathrooms=1, Square_Area=85.0,  Price=65000.0),
    Row(City="Amman",    Bedrooms=4, Bathrooms=3, Square_Area=220.0, Price=185000.0),
    Row(City="Aqaba",    Bedrooms=2, Bathrooms=2, Square_Area=100.0, Price=88000.0),
    Row(City="Amman",    Bedrooms=3, Bathrooms=2, Square_Area=155.0, Price=128000.0),
    Row(City="Zarqa",    Bedrooms=4, Bathrooms=2, Square_Area=160.0, Price=105000.0),
    Row(City="Irbid",    Bedrooms=3, Bathrooms=2, Square_Area=145.0, Price=98000.0),
    Row(City="Amman",    Bedrooms=2, Bathrooms=1, Square_Area=80.0,  Price=62000.0),
    Row(City="Aqaba",    Bedrooms=4, Bathrooms=3, Square_Area=180.0, Price=145000.0),
    Row(City="Amman",    Bedrooms=6, Bathrooms=5, Square_Area=380.0, Price=320000.0),
    Row(City="Zarqa",    Bedrooms=1, Bathrooms=1, Square_Area=55.0,  Price=40000.0),
    Row(City="Irbid",    Bedrooms=5, Bathrooms=3, Square_Area=250.0, Price=190000.0),
    Row(City="Amman",    Bedrooms=3, Bathrooms=3, Square_Area=170.0, Price=142000.0),
    Row(City="Aqaba",    Bedrooms=1, Bathrooms=1, Square_Area=50.0,  Price=38000.0),
    Row(City="Amman",    Bedrooms=4, Bathrooms=2, Square_Area=190.0, Price=155000.0),
    Row(City="Zarqa",    Bedrooms=3, Bathrooms=1, Square_Area=120.0, Price=82000.0),
    Row(City="Irbid",    Bedrooms=2, Bathrooms=2, Square_Area=95.0,  Price=72000.0),
    Row(City="Amman",    Bedrooms=2, Bathrooms=2, Square_Area=110.0, Price=90000.0),
    Row(City="Aqaba",    Bedrooms=3, Bathrooms=2, Square_Area=135.0, Price=108000.0),
]

df = spark.createDataFrame(data)

print(f"Number of rows   : {df.count()}")
print(f"Number of columns: {len(df.columns)}")
print(f"Columns          : {df.columns}")

Number of rows   : 25
Number of columns: 5
Columns          : ['City', 'Bedrooms', 'Bathrooms', 'Square_Area', 'Price']


In [43]:
# printSchema() — shows column names, data types, and nullability
df.printSchema()

root
 |-- City: string (nullable = true)
 |-- Bedrooms: long (nullable = true)
 |-- Bathrooms: long (nullable = true)
 |-- Square_Area: double (nullable = true)
 |-- Price: double (nullable = true)



In [44]:
# show() — displays first n rows in tabular format
df.show(10)

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|       90.0| 70000.0|
|Irbid|       4|        3|      200.0|160000.0|
|Amman|       1|        1|       60.0| 50000.0|
|Aqaba|       3|        2|      140.0|115000.0|
|Amman|       5|        4|      300.0|250000.0|
|Zarqa|       3|        2|      130.0| 95000.0|
|Irbid|       2|        1|       85.0| 65000.0|
|Amman|       4|        3|      220.0|185000.0|
|Aqaba|       2|        2|      100.0| 88000.0|
+-----+--------+---------+-----------+--------+
only showing top 10 rows


In [45]:
# describe() — summary statistics for numeric (and string) columns
df.describe().show()

+-------+-----+------------------+------------------+----------------+-----------------+
|summary| City|          Bedrooms|         Bathrooms|     Square_Area|            Price|
+-------+-----+------------------+------------------+----------------+-----------------+
|  count|   25|                25|                25|              25|               25|
|   mean| NULL|               3.0|              2.12|           150.0|         118920.0|
| stddev| NULL|1.2909944487358056|1.0132456102380443|77.5806032459145|65853.95457626924|
|    min|Amman|                 1|                 1|            50.0|          38000.0|
|    max|Zarqa|                 6|                 5|           380.0|         320000.0|
+-------+-----+------------------+------------------+----------------+-----------------+



**Expected output of `describe()`:**  
A table showing `count`, `mean`, `stddev`, `min`, and `max` for each column. For numeric columns this gives a quick statistical profile of the dataset.

### 3.2 From a List of Tuples with an Explicit Schema

Use `StructType` + `StructField` to define a precise schema, controlling column names, data types, and whether nulls are allowed.

In [46]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("City",        StringType(),  nullable=False),
    StructField("Bedrooms",    IntegerType(), nullable=False),
    StructField("Bathrooms",   IntegerType(), nullable=False),
    StructField("Square_Area", DoubleType(),  nullable=True),
    StructField("Price",       DoubleType(),  nullable=True),
])

tuple_data = [
    ("Amman", 3, 2, 150.0, 120000.0),
    ("Zarqa", 2, 1,  90.0,  70000.0),
    ("Irbid", 4, 3, 200.0, 160000.0),
    ("Aqaba", 3, 2, 140.0, 115000.0),
]

df_typed = spark.createDataFrame(tuple_data, schema=schema)
df_typed.printSchema()
df_typed.show()

root
 |-- City: string (nullable = false)
 |-- Bedrooms: integer (nullable = false)
 |-- Bathrooms: integer (nullable = false)
 |-- Square_Area: double (nullable = true)
 |-- Price: double (nullable = true)

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|       90.0| 70000.0|
|Irbid|       4|        3|      200.0|160000.0|
|Aqaba|       3|        2|      140.0|115000.0|
+-----+--------+---------+-----------+--------+



### 3.3 From a CSV File

In a real project you would load `apartment_prices.csv` from disk or cloud storage. The code below first writes a CSV from our in-memory data, then reads it back — demonstrating a full round-trip.

In [47]:
import os

# --- Write a sample CSV so we have a file to read ---
csv_path = "/tmp/apartment_prices.csv"
df.toPandas().to_csv(csv_path, index=False)
print(f"CSV written to: {csv_path}")

# --- Read CSV with spark.read.csv ---
df_csv = spark.read.csv(
    csv_path,
    header=True,       # first row is the header
    inferSchema=True,  # automatically detect column types
    sep=","            # delimiter (default is comma)
)

print("\nSchema inferred from CSV:")
df_csv.printSchema()
df_csv.show(5)

CSV written to: /tmp/apartment_prices.csv

Schema inferred from CSV:
root
 |-- City: string (nullable = true)
 |-- Bedrooms: integer (nullable = true)
 |-- Bathrooms: integer (nullable = true)
 |-- Square_Area: double (nullable = true)
 |-- Price: double (nullable = true)

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|       90.0| 70000.0|
|Irbid|       4|        3|      200.0|160000.0|
|Amman|       1|        1|       60.0| 50000.0|
|Aqaba|       3|        2|      140.0|115000.0|
+-----+--------+---------+-----------+--------+
only showing top 5 rows


**Common `spark.read.csv` options:**

| Option | Description | Example |
|---|---|---|
| `header` | First row contains column names | `True` / `False` |
| `inferSchema` | Auto-detect column types (slower) | `True` / `False` |
| `sep` | Field delimiter | `","` / `";"` / `"\t"` |
| `nullValue` | String to treat as null | `"NA"` / `""` |
| `encoding` | File character encoding | `"UTF-8"` |
| `multiLine` | Allow multi-line field values | `True` |

> **Tip:** Always specify the schema explicitly in production to avoid the cost of the schema-inference pass and to prevent type mismatches.

### 3.4 From JSON

In [48]:
import json

# Write a JSON lines file (one JSON object per line)
json_path = "/tmp/apartment_prices.json"
records = df.toPandas().to_dict(orient="records")
with open(json_path, "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

# Read JSON
df_json = spark.read.json(json_path)
df_json.printSchema()
df_json.show(5)

root
 |-- Bathrooms: long (nullable = true)
 |-- Bedrooms: long (nullable = true)
 |-- City: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Square_Area: double (nullable = true)

+---------+--------+-----+--------+-----------+
|Bathrooms|Bedrooms| City|   Price|Square_Area|
+---------+--------+-----+--------+-----------+
|        2|       3|Amman|120000.0|      150.0|
|        1|       2|Zarqa| 70000.0|       90.0|
|        3|       4|Irbid|160000.0|      200.0|
|        1|       1|Amman| 50000.0|       60.0|
|        2|       3|Aqaba|115000.0|      140.0|
+---------+--------+-----+--------+-----------+
only showing top 5 rows


<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  4. DataFrame API — Core Operations
</h2>

The DataFrame API provides a rich set of transformations. All transformations are **lazy** — they are not executed until an **action** (like `show()`, `count()`, or `collect()`) is called.

### 4.1 `select()` — Selecting Columns

In [49]:
from pyspark.sql.functions import col

# Select specific columns by name (string)
df.select("City", "Price").show(5)

# Select using col() — more powerful (supports expressions)
df.select(col("City"), col("Bedrooms"), col("Price")).show(5)

# Select with an expression — price in thousands
df.select(
    "City",
    "Price",
    (col("Price") / 1000).alias("Price_K")
).show(5)

+-----+--------+
| City|   Price|
+-----+--------+
|Amman|120000.0|
|Zarqa| 70000.0|
|Irbid|160000.0|
|Amman| 50000.0|
|Aqaba|115000.0|
+-----+--------+
only showing top 5 rows
+-----+--------+--------+
| City|Bedrooms|   Price|
+-----+--------+--------+
|Amman|       3|120000.0|
|Zarqa|       2| 70000.0|
|Irbid|       4|160000.0|
|Amman|       1| 50000.0|
|Aqaba|       3|115000.0|
+-----+--------+--------+
only showing top 5 rows
+-----+--------+-------+
| City|   Price|Price_K|
+-----+--------+-------+
|Amman|120000.0|  120.0|
|Zarqa| 70000.0|   70.0|
|Irbid|160000.0|  160.0|
|Amman| 50000.0|   50.0|
|Aqaba|115000.0|  115.0|
+-----+--------+-------+
only showing top 5 rows


### 4.2 `filter()` / `where()` — Filtering Rows

`filter()` and `where()` are aliases — both do the same thing.

In [50]:
# Filter with a Python expression string
df.filter("City = 'Amman'").show(5)

# Filter with col() expression
df.filter(col("Price") > 100000).show(5)

# Multiple conditions: use & (AND), | (OR), ~ (NOT)
df.filter((col("City") == "Amman") & (col("Bedrooms") >= 3)).show()

# where() is identical to filter()
df.where(col("Bathrooms") == 2).show(5)

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Amman|       1|        1|       60.0| 50000.0|
|Amman|       5|        4|      300.0|250000.0|
|Amman|       4|        3|      220.0|185000.0|
|Amman|       3|        2|      155.0|128000.0|
+-----+--------+---------+-----------+--------+
only showing top 5 rows
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Irbid|       4|        3|      200.0|160000.0|
|Aqaba|       3|        2|      140.0|115000.0|
|Amman|       5|        4|      300.0|250000.0|
|Amman|       4|        3|      220.0|185000.0|
+-----+--------+---------+-----------+--------+
only showing top 5 rows
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|  

> **Pitfall:** Always wrap each condition in parentheses when combining with `&` or `|`. Operator precedence in Python can cause unexpected results otherwise.

### 4.3 `withColumn()` — Adding or Transforming Columns

In [51]:
from pyspark.sql.functions import round as spark_round

# Add a new column: price per square meter
df2 = df.withColumn("Price_per_sqm", spark_round(col("Price") / col("Square_Area"), 2))

# Transform an existing column: convert price to USD thousands
df2 = df2.withColumn("Price_K", spark_round(col("Price") / 1000, 1))

df2.select("City", "Square_Area", "Price", "Price_per_sqm", "Price_K").show(8)

+-----+-----------+--------+-------------+-------+
| City|Square_Area|   Price|Price_per_sqm|Price_K|
+-----+-----------+--------+-------------+-------+
|Amman|      150.0|120000.0|        800.0|  120.0|
|Zarqa|       90.0| 70000.0|       777.78|   70.0|
|Irbid|      200.0|160000.0|        800.0|  160.0|
|Amman|       60.0| 50000.0|       833.33|   50.0|
|Aqaba|      140.0|115000.0|       821.43|  115.0|
|Amman|      300.0|250000.0|       833.33|  250.0|
|Zarqa|      130.0| 95000.0|       730.77|   95.0|
|Irbid|       85.0| 65000.0|       764.71|   65.0|
+-----+-----------+--------+-------------+-------+
only showing top 8 rows


### 4.4 `drop()` and `withColumnRenamed()` — Removing and Renaming Columns

In [52]:
# Drop columns
df_dropped = df2.drop("Price_K", "Price_per_sqm")
print("Columns after drop:", df_dropped.columns)

# Rename a column
df_renamed = df.withColumnRenamed("Square_Area", "Area_sqm") \
               .withColumnRenamed("Price", "Price_USD")
print("Columns after rename:", df_renamed.columns)
df_renamed.show(3)

Columns after drop: ['City', 'Bedrooms', 'Bathrooms', 'Square_Area', 'Price']
Columns after rename: ['City', 'Bedrooms', 'Bathrooms', 'Area_sqm', 'Price_USD']
+-----+--------+---------+--------+---------+
| City|Bedrooms|Bathrooms|Area_sqm|Price_USD|
+-----+--------+---------+--------+---------+
|Amman|       3|        2|   150.0| 120000.0|
|Zarqa|       2|        1|    90.0|  70000.0|
|Irbid|       4|        3|   200.0| 160000.0|
+-----+--------+---------+--------+---------+
only showing top 3 rows


### 4.5 `orderBy()` / `sort()` — Sorting Rows

In [53]:
from pyspark.sql.functions import desc, asc

# Sort by Price descending (most expensive first)
df.orderBy(desc("Price")).show(5)

# Sort by City ascending, then Price descending
df.orderBy(asc("City"), desc("Price")).show(8)

# Alternative syntax
df.sort(col("Price").desc()).show(5)

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       6|        5|      380.0|320000.0|
|Amman|       5|        4|      300.0|250000.0|
|Irbid|       5|        3|      250.0|190000.0|
|Amman|       4|        3|      220.0|185000.0|
|Irbid|       4|        3|      200.0|160000.0|
+-----+--------+---------+-----------+--------+
only showing top 5 rows
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       6|        5|      380.0|320000.0|
|Amman|       5|        4|      300.0|250000.0|
|Amman|       4|        3|      220.0|185000.0|
|Amman|       4|        2|      190.0|155000.0|
|Amman|       3|        3|      170.0|142000.0|
|Amman|       3|        2|      155.0|128000.0|
|Amman|       3|        2|      150.0|120000.0|
|Amman|       2|        2|      110.0| 90000.0|
+-----+--------+

### 4.6 `limit()` — Limiting the Number of Rows

In [54]:
# Return only the top 5 rows (as a new DataFrame)
df_top5 = df.orderBy(desc("Price")).limit(5)
df_top5.show()

print(f"df_top5 row count: {df_top5.count()}")

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       6|        5|      380.0|320000.0|
|Amman|       5|        4|      300.0|250000.0|
|Irbid|       5|        3|      250.0|190000.0|
|Amman|       4|        3|      220.0|185000.0|
|Irbid|       4|        3|      200.0|160000.0|
+-----+--------+---------+-----------+--------+

df_top5 row count: 5


<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  5. Aggregations with the DataFrame API
</h2>

Aggregations summarize data by group. The pattern is always `groupBy(...).agg(...)`.

In [55]:
from pyspark.sql.functions import count, avg, sum as spark_sum, min as spark_min, max as spark_max

# Count apartments per city
print("=== Apartment count per city ===")
df.groupBy("City").count().orderBy(desc("count")).show()

# Average price per city
print("=== Average price per city ===")
df.groupBy("City").avg("Price").show()

=== Apartment count per city ===
+-----+-----+
| City|count|
+-----+-----+
|Amman|   10|
|Irbid|    5|
|Aqaba|    5|
|Zarqa|    5|
+-----+-----+

=== Average price per city ===
+-----+----------+
| City|avg(Price)|
+-----+----------+
|Amman|  150200.0|
|Irbid|  117000.0|
|Aqaba|   98800.0|
|Zarqa|   78400.0|
+-----+----------+



In [56]:
# Multiple aggregations at once using agg()
agg_df = df.groupBy("City").agg(
    count("*").alias("num_apartments"),
    spark_round(avg("Price"), 2).alias("avg_price"),
    spark_min("Price").alias("min_price"),
    spark_max("Price").alias("max_price"),
    spark_round(avg("Square_Area"), 1).alias("avg_area_sqm")
)

agg_df.orderBy(desc("avg_price")).show()

+-----+--------------+---------+---------+---------+------------+
| City|num_apartments|avg_price|min_price|max_price|avg_area_sqm|
+-----+--------------+---------+---------+---------+------------+
|Amman|            10| 150200.0|  50000.0| 320000.0|       181.5|
|Irbid|             5| 117000.0|  65000.0| 190000.0|       155.0|
|Aqaba|             5|  98800.0|  38000.0| 145000.0|       121.0|
|Zarqa|             5|  78400.0|  40000.0| 105000.0|       111.0|
+-----+--------------+---------+---------+---------+------------+



In [57]:
# Group by multiple columns: City + Bedrooms
df.groupBy("City", "Bedrooms").agg(
    count("*").alias("count"),
    spark_round(avg("Price"), 0).alias("avg_price")
).orderBy("City", "Bedrooms").show(15)

+-----+--------+-----+---------+
| City|Bedrooms|count|avg_price|
+-----+--------+-----+---------+
|Amman|       1|    1|  50000.0|
|Amman|       2|    2|  76000.0|
|Amman|       3|    3| 130000.0|
|Amman|       4|    2| 170000.0|
|Amman|       5|    1| 250000.0|
|Amman|       6|    1| 320000.0|
|Aqaba|       1|    1|  38000.0|
|Aqaba|       2|    1|  88000.0|
|Aqaba|       3|    2| 111500.0|
|Aqaba|       4|    1| 145000.0|
|Irbid|       2|    2|  68500.0|
|Irbid|       3|    1|  98000.0|
|Irbid|       4|    1| 160000.0|
|Irbid|       5|    1| 190000.0|
|Zarqa|       1|    1|  40000.0|
+-----+--------+-----+---------+
only showing top 15 rows


**Expected output:** A table with one row per `(City, Bedrooms)` combination, showing the listing count and average price. This lets you compare how bedroom count affects price across cities.

### 5.4 Reading the Query Plan with

Every Spark DataFrame query is compiled into an **execution plan** by the Catalyst Optimizer before a single byte of data is read or shuffled. Calling  prints that plan in four layers:

| Layer | What it shows |
|---|---|
| **Parsed Logical Plan** | Raw plan — unresolved column names |
| **Analyzed Logical Plan** | Column names resolved, types checked |
| **Optimized Logical Plan** | Catalyst-optimized: predicates pushed down, projections pruned |
| **Physical Plan** | Actual execution strategy (sort-merge join, broadcast join, exchange) |

> **Why bother?** When a query runs slower than expected,  shows you *exactly* why — e.g., a missing predicate pushdown or an unintended shuffle ().


In [58]:
# Filtered aggregation: average price per city for large apartments
query = (
    df
    .filter(col("Square_Area") > 120)
    .groupBy("City")
    .agg(avg("Price").alias("avg_price_large"))
    .orderBy(desc("avg_price_large"))
)

query.show()

# Print the full query plan (parsed → analyzed → optimized → physical)
print("=== FULL QUERY PLAN ===")
query.explain(True)

# Key things to look for in the physical plan:
#   - Filter appears BEFORE Aggregate  → predicate pushed down (good)
#   - Exchange hashpartitioning(...)   → a shuffle is happening (expensive)
#   - No Exchange before Filter        → filter applied per partition (cheap)


+-----+------------------+
| City|   avg_price_large|
+-----+------------------+
|Amman| 185714.2857142857|
|Irbid|149333.33333333334|
|Aqaba|122666.66666666667|
|Zarqa|          100000.0|
+-----+------------------+

=== FULL QUERY PLAN ===
== Parsed Logical Plan ==
'Sort ['avg_price_large DESC NULLS LAST], true
+- Aggregate [City#1605], [City#1605, avg(Price#1609) AS avg_price_large#2490]
   +- Filter (Square_Area#1608 > cast(120 as double))
      +- LogicalRDD [City#1605, Bedrooms#1606L, Bathrooms#1607L, Square_Area#1608, Price#1609], false

== Analyzed Logical Plan ==
City: string, avg_price_large: double
Sort [avg_price_large#2490 DESC NULLS LAST], true
+- Aggregate [City#1605], [City#1605, avg(Price#1609) AS avg_price_large#2490]
   +- Filter (Square_Area#1608 > cast(120 as double))
      +- LogicalRDD [City#1605, Bedrooms#1606L, Bathrooms#1607L, Square_Area#1608, Price#1609], false

== Optimized Logical Plan ==
Sort [avg_price_large#2490 DESC NULLS LAST], true
+- Aggregate [City#

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  6. Joins
</h2>

Spark supports the same join types as SQL. Here we join our apartments DataFrame with a small `city_info` reference table.

In [59]:
# Create a reference DataFrame with city metadata
city_info = spark.createDataFrame([
    Row(City="Amman",  Region="Central",  Population=4000000),
    Row(City="Zarqa",  Region="Central",  Population=1000000),
    Row(City="Irbid",  Region="Northern", Population=660000),
    Row(City="Aqaba",  Region="Southern", Population=188000),
    Row(City="Madaba", Region="Central",  Population=85000),  # no apartments in main df
])

city_info.show()

+------+--------+----------+
|  City|  Region|Population|
+------+--------+----------+
| Amman| Central|   4000000|
| Zarqa| Central|   1000000|
| Irbid|Northern|    660000|
| Aqaba|Southern|    188000|
|Madaba| Central|     85000|
+------+--------+----------+



In [60]:
# INNER JOIN — only rows with a match in both DataFrames
print("=== INNER JOIN ===")
inner_join = df.join(city_info, on="City", how="inner")
inner_join.select("City", "Region", "Population", "Bedrooms", "Price").show(8)

=== INNER JOIN ===
+-----+-------+----------+--------+--------+
| City| Region|Population|Bedrooms|   Price|
+-----+-------+----------+--------+--------+
|Amman|Central|   4000000|       3|120000.0|
|Amman|Central|   4000000|       1| 50000.0|
|Amman|Central|   4000000|       5|250000.0|
|Amman|Central|   4000000|       4|185000.0|
|Amman|Central|   4000000|       3|128000.0|
|Amman|Central|   4000000|       2| 62000.0|
|Amman|Central|   4000000|       6|320000.0|
|Amman|Central|   4000000|       3|142000.0|
+-----+-------+----------+--------+--------+
only showing top 8 rows


In [61]:
# LEFT JOIN — all rows from the left (apartments), matched rows from right (city_info)
print("=== LEFT JOIN ===")
left_join = df.join(city_info, on="City", how="left")
left_join.select("City", "Region", "Population", "Price").show(8)

# RIGHT JOIN — all rows from city_info, matched rows from apartments
print("=== RIGHT JOIN (includes Madaba with no apartments) ===")
right_join = df.join(city_info, on="City", how="right")
right_join.groupBy("City", "Region").count().orderBy("City").show()

=== LEFT JOIN ===
+-----+--------+----------+--------+
| City|  Region|Population|   Price|
+-----+--------+----------+--------+
|Amman| Central|   4000000|120000.0|
|Amman| Central|   4000000| 50000.0|
|Amman| Central|   4000000|250000.0|
|Amman| Central|   4000000|185000.0|
|Irbid|Northern|    660000|160000.0|
|Aqaba|Southern|    188000|115000.0|
|Irbid|Northern|    660000| 65000.0|
|Zarqa| Central|   1000000| 70000.0|
+-----+--------+----------+--------+
only showing top 8 rows
=== RIGHT JOIN (includes Madaba with no apartments) ===
+------+--------+-----+
|  City|  Region|count|
+------+--------+-----+
| Amman| Central|   10|
| Aqaba|Southern|    5|
| Irbid|Northern|    5|
|Madaba| Central|    1|
| Zarqa| Central|    5|
+------+--------+-----+



| Join Type | Description |
|---|---|
| `inner` | Only matching rows from both sides |
| `left` (left_outer) | All rows from left + matching from right (nulls for non-matches) |
| `right` (right_outer) | All rows from right + matching from left |
| `full` (full_outer) | All rows from both sides (nulls where no match) |
| `cross` | Cartesian product — every row paired with every row |
| `semi` | Rows from left where a match exists in right (no right columns) |
| `anti` | Rows from left where NO match exists in right |

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  7. Built-in Functions
</h2>

`pyspark.sql.functions` provides hundreds of built-in functions. Always prefer these over Python UDFs because they run natively in the JVM and benefit from Catalyst optimization.

### 7.1 String Functions

In [62]:
from pyspark.sql.functions import (
    upper, lower, length, concat, concat_ws, split, trim, lit
)

df_str = df.select(
    col("City"),
    upper(col("City")).alias("City_Upper"),
    lower(col("City")).alias("City_Lower"),
    length(col("City")).alias("City_Length"),
    concat(col("City"), lit(" - "), col("Bedrooms").cast("string"), lit(" BR")).alias("Label"),
)

df_str.show(8, truncate=False)

+-----+----------+----------+-----------+------------+
|City |City_Upper|City_Lower|City_Length|Label       |
+-----+----------+----------+-----------+------------+
|Amman|AMMAN     |amman     |5          |Amman - 3 BR|
|Zarqa|ZARQA     |zarqa     |5          |Zarqa - 2 BR|
|Irbid|IRBID     |irbid     |5          |Irbid - 4 BR|
|Amman|AMMAN     |amman     |5          |Amman - 1 BR|
|Aqaba|AQABA     |aqaba     |5          |Aqaba - 3 BR|
|Amman|AMMAN     |amman     |5          |Amman - 5 BR|
|Zarqa|ZARQA     |zarqa     |5          |Zarqa - 3 BR|
|Irbid|IRBID     |irbid     |5          |Irbid - 2 BR|
+-----+----------+----------+-----------+------------+
only showing top 8 rows


### 7.2 Numeric Functions

In [63]:
from pyspark.sql.functions import round as spark_round, abs as spark_abs, sqrt, log, pow as spark_pow

df_num = df.select(
    "City", "Square_Area", "Price",
    spark_round(col("Price") / col("Square_Area"), 2).alias("Price_per_sqm"),
    spark_round(sqrt(col("Square_Area")), 2).alias("Sqrt_Area"),
    spark_round(log(col("Price")), 4).alias("Log_Price"),
)

df_num.show(6)

+-----+-----------+--------+-------------+---------+---------+
| City|Square_Area|   Price|Price_per_sqm|Sqrt_Area|Log_Price|
+-----+-----------+--------+-------------+---------+---------+
|Amman|      150.0|120000.0|        800.0|    12.25|  11.6952|
|Zarqa|       90.0| 70000.0|       777.78|     9.49|  11.1563|
|Irbid|      200.0|160000.0|        800.0|    14.14|  11.9829|
|Amman|       60.0| 50000.0|       833.33|     7.75|  10.8198|
|Aqaba|      140.0|115000.0|       821.43|    11.83|  11.6527|
|Amman|      300.0|250000.0|       833.33|    17.32|  12.4292|
+-----+-----------+--------+-------------+---------+---------+
only showing top 6 rows


### 7.3 Date Functions

In [64]:
from pyspark.sql.functions import current_date, current_timestamp, year, month, dayofmonth, date_format

df_dates = df.limit(4).select(
    "City",
    current_date().alias("today"),
    current_timestamp().alias("now"),
    year(current_date()).alias("year"),
    month(current_date()).alias("month"),
    dayofmonth(current_date()).alias("day"),
    date_format(current_date(), "yyyy-MM-dd").alias("formatted_date"),
)

df_dates.show(truncate=False)

+-----+----------+--------------------------+----+-----+---+--------------+
|City |today     |now                       |year|month|day|formatted_date|
+-----+----------+--------------------------+----+-----+---+--------------+
|Amman|2026-04-27|2026-04-27 04:53:15.079426|2026|4    |27 |2026-04-27    |
|Zarqa|2026-04-27|2026-04-27 04:53:15.079426|2026|4    |27 |2026-04-27    |
|Irbid|2026-04-27|2026-04-27 04:53:15.079426|2026|4    |27 |2026-04-27    |
|Amman|2026-04-27|2026-04-27 04:53:15.079426|2026|4    |27 |2026-04-27    |
+-----+----------+--------------------------+----+-----+---+--------------+



### 7.4 Conditional Functions: `when()` / `otherwise()`

These are the Spark equivalent of `CASE WHEN ... THEN ... ELSE ... END` in SQL.

In [65]:
from pyspark.sql.functions import when

# Classify apartments by price tier
df_tier = df.withColumn(
    "Price_Tier",
    when(col("Price") < 70000,  "Budget")
    .when(col("Price") < 130000, "Mid-range")
    .when(col("Price") < 200000, "Premium")
    .otherwise("Luxury")
)

# Also classify by size
df_tier = df_tier.withColumn(
    "Size_Category",
    when(col("Square_Area") < 80,  "Small")
    .when(col("Square_Area") < 150, "Medium")
    .otherwise("Large")
)

df_tier.select("City", "Bedrooms", "Square_Area", "Price", "Price_Tier", "Size_Category").show(10)

+-----+--------+-----------+--------+----------+-------------+
| City|Bedrooms|Square_Area|   Price|Price_Tier|Size_Category|
+-----+--------+-----------+--------+----------+-------------+
|Amman|       3|      150.0|120000.0| Mid-range|        Large|
|Zarqa|       2|       90.0| 70000.0| Mid-range|       Medium|
|Irbid|       4|      200.0|160000.0|   Premium|        Large|
|Amman|       1|       60.0| 50000.0|    Budget|        Small|
|Aqaba|       3|      140.0|115000.0| Mid-range|       Medium|
|Amman|       5|      300.0|250000.0|    Luxury|        Large|
|Zarqa|       3|      130.0| 95000.0| Mid-range|       Medium|
|Irbid|       2|       85.0| 65000.0|    Budget|       Medium|
|Amman|       4|      220.0|185000.0|   Premium|        Large|
|Aqaba|       2|      100.0| 88000.0| Mid-range|       Medium|
+-----+--------+-----------+--------+----------+-------------+
only showing top 10 rows


In [66]:
# Distribution of price tiers
df_tier.groupBy("Price_Tier").count().orderBy("Price_Tier").show()

+----------+-----+
|Price_Tier|count|
+----------+-----+
|    Budget|    5|
|    Luxury|    2|
| Mid-range|   12|
|   Premium|    6|
+----------+-----+



<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  8. Handling Missing Data
</h2>

Real datasets always contain missing values. Spark provides `dropna()` and `fillna()` for handling them.

In [67]:
from pyspark.sql.functions import isnan, isnull, when

# Introduce some missing values for demonstration
null_data = [
    Row(City="Amman",  Bedrooms=3, Bathrooms=2, Square_Area=150.0,  Price=120000.0),
    Row(City="Zarqa",  Bedrooms=2, Bathrooms=1, Square_Area=None,   Price=70000.0),
    Row(City=None,     Bedrooms=4, Bathrooms=3, Square_Area=200.0,  Price=160000.0),
    Row(City="Amman",  Bedrooms=1, Bathrooms=1, Square_Area=60.0,   Price=None),
    Row(City="Aqaba",  Bedrooms=3, Bathrooms=2, Square_Area=None,   Price=None),
    Row(City="Irbid",  Bedrooms=2, Bathrooms=1, Square_Area=85.0,   Price=65000.0),
]

df_null = spark.createDataFrame(null_data)
df_null.show()

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|       NULL| 70000.0|
| NULL|       4|        3|      200.0|160000.0|
|Amman|       1|        1|       60.0|    NULL|
|Aqaba|       3|        2|       NULL|    NULL|
|Irbid|       2|        1|       85.0| 65000.0|
+-----+--------+---------+-----------+--------+



In [68]:
from pyspark.sql.functions import col, sum as spark_sum

# Count nulls per column
print("=== Null count per column ===")
null_counts = df_null.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_null.columns
])
null_counts.show()

=== Null count per column ===
+----+--------+---------+-----------+-----+
|City|Bedrooms|Bathrooms|Square_Area|Price|
+----+--------+---------+-----------+-----+
|   1|       0|        0|          2|    2|
+----+--------+---------+-----------+-----+



In [69]:
# dropna() — remove rows with ANY null value
print(f"Original rows  : {df_null.count()}")
print(f"After dropna() : {df_null.dropna().count()}")
df_null.dropna().show()

# Drop rows only if specific columns are null
print("\nDrop rows where Price is null:")
df_null.dropna(subset=["Price"]).show()

# Drop rows where at least 'thresh' non-null values exist
print("\nKeep rows with at least 4 non-null values:")
df_null.dropna(thresh=4).show()

Original rows  : 6
After dropna() : 2
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Irbid|       2|        1|       85.0| 65000.0|
+-----+--------+---------+-----------+--------+


Drop rows where Price is null:
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|       NULL| 70000.0|
| NULL|       4|        3|      200.0|160000.0|
|Irbid|       2|        1|       85.0| 65000.0|
+-----+--------+---------+-----------+--------+


Keep rows with at least 4 non-null values:
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|    

In [70]:
# fillna() — replace nulls with a specified value

# Fill all nulls with a single value
df_null.fillna(0).show()

# Fill specific columns with different values
df_filled = df_null.fillna({
    "City":        "Unknown",
    "Square_Area": 100.0,
    "Price":       0.0
})

print("\nAfter targeted fillna():")
df_filled.show()

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|        0.0| 70000.0|
| NULL|       4|        3|      200.0|160000.0|
|Amman|       1|        1|       60.0|     0.0|
|Aqaba|       3|        2|        0.0|     0.0|
|Irbid|       2|        1|       85.0| 65000.0|
+-----+--------+---------+-----------+--------+


After targeted fillna():
+-------+--------+---------+-----------+--------+
|   City|Bedrooms|Bathrooms|Square_Area|   Price|
+-------+--------+---------+-----------+--------+
|  Amman|       3|        2|      150.0|120000.0|
|  Zarqa|       2|        1|      100.0| 70000.0|
|Unknown|       4|        3|      200.0|160000.0|
|  Amman|       1|        1|       60.0|     0.0|
|  Aqaba|       3|        2|      100.0|     0.0|
|  Irbid|       2|        1|       85.0| 65000.0|
+-------+--------+---------+-----------+---

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  9. Spark SQL
</h2>

Spark SQL lets you run standard SQL queries on DataFrames. You must first register the DataFrame as a **temporary view**. The DataFrame API and Spark SQL are equivalent — they compile to the same execution plan.

### 9.1 Registering Temporary Views

In [71]:
# Register main apartments DataFrame as a SQL view
df.createOrReplaceTempView("apartments")

# Register city_info for joins
city_info.createOrReplaceTempView("city_info")

# Register the tier-enriched DataFrame
df_tier.createOrReplaceTempView("apartments_tier")

# Verify views are registered
spark.sql("SHOW VIEWS").show()

+---------+---------------+-----------+
|namespace|       viewName|isTemporary|
+---------+---------------+-----------+
|         |     apartments|       true|
|         |apartments_tier|       true|
|         |      city_info|       true|
+---------+---------------+-----------+



### 9.2 Basic SELECT, WHERE, ORDER BY, LIMIT

In [72]:
# Basic SELECT
spark.sql("""
    SELECT City, Bedrooms, Bathrooms, Square_Area, Price
    FROM apartments
    LIMIT 10
""").show()

# WHERE clause
spark.sql("""
    SELECT City, Bedrooms, Price
    FROM apartments
    WHERE City = 'Amman'
      AND Price > 100000
    ORDER BY Price DESC
""").show()

# Computed column
spark.sql("""
    SELECT
        City,
        Price,
        Square_Area,
        ROUND(Price / Square_Area, 2) AS price_per_sqm
    FROM apartments
    ORDER BY price_per_sqm DESC
    LIMIT 10
""").show()

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       3|        2|      150.0|120000.0|
|Zarqa|       2|        1|       90.0| 70000.0|
|Irbid|       4|        3|      200.0|160000.0|
|Amman|       1|        1|       60.0| 50000.0|
|Aqaba|       3|        2|      140.0|115000.0|
|Amman|       5|        4|      300.0|250000.0|
|Zarqa|       3|        2|      130.0| 95000.0|
|Irbid|       2|        1|       85.0| 65000.0|
|Amman|       4|        3|      220.0|185000.0|
|Aqaba|       2|        2|      100.0| 88000.0|
+-----+--------+---------+-----------+--------+

+-----+--------+--------+
| City|Bedrooms|   Price|
+-----+--------+--------+
|Amman|       6|320000.0|
|Amman|       5|250000.0|
|Amman|       4|185000.0|
|Amman|       4|155000.0|
|Amman|       3|142000.0|
|Amman|       3|128000.0|
|Amman|       3|120000.0|
+-----+--------+--------+

+-----+--------+-----------+------------

### 9.3 GROUP BY with Aggregation Functions

In [73]:
spark.sql("""
    SELECT
        City,
        COUNT(*)                    AS total_listings,
        ROUND(AVG(Price), 0)        AS avg_price,
        MIN(Price)                  AS min_price,
        MAX(Price)                  AS max_price,
        ROUND(AVG(Square_Area), 1)  AS avg_area
    FROM apartments
    GROUP BY City
    ORDER BY avg_price DESC
""").show()

+-----+--------------+---------+---------+---------+--------+
| City|total_listings|avg_price|min_price|max_price|avg_area|
+-----+--------------+---------+---------+---------+--------+
|Amman|            10| 150200.0|  50000.0| 320000.0|   181.5|
|Irbid|             5| 117000.0|  65000.0| 190000.0|   155.0|
|Aqaba|             5|  98800.0|  38000.0| 145000.0|   121.0|
|Zarqa|             5|  78400.0|  40000.0| 105000.0|   111.0|
+-----+--------------+---------+---------+---------+--------+



### 9.4 JOINs in SQL

In [74]:
# INNER JOIN apartments with city_info
spark.sql("""
    SELECT
        a.City,
        ci.Region,
        ci.Population,
        a.Bedrooms,
        a.Price,
        ROUND(a.Price / ci.Population, 4) AS price_per_resident
    FROM apartments a
    INNER JOIN city_info ci ON a.City = ci.City
    ORDER BY a.Price DESC
    LIMIT 8
""").show()

# RIGHT JOIN to see all cities, including Madaba with no apartments
spark.sql("""
    SELECT
        ci.City,
        ci.Region,
        COUNT(a.City) AS listing_count
    FROM apartments a
    RIGHT JOIN city_info ci ON a.City = ci.City
    GROUP BY ci.City, ci.Region
    ORDER BY listing_count DESC
""").show()

+-----+--------+----------+--------+--------+------------------+
| City|  Region|Population|Bedrooms|   Price|price_per_resident|
+-----+--------+----------+--------+--------+------------------+
|Amman| Central|   4000000|       6|320000.0|              0.08|
|Amman| Central|   4000000|       5|250000.0|            0.0625|
|Irbid|Northern|    660000|       5|190000.0|            0.2879|
|Amman| Central|   4000000|       4|185000.0|            0.0463|
|Irbid|Northern|    660000|       4|160000.0|            0.2424|
|Amman| Central|   4000000|       4|155000.0|            0.0388|
|Aqaba|Southern|    188000|       4|145000.0|            0.7713|
|Amman| Central|   4000000|       3|142000.0|            0.0355|
+-----+--------+----------+--------+--------+------------------+

+------+--------+-------------+
|  City|  Region|listing_count|
+------+--------+-------------+
| Amman| Central|           10|
| Zarqa| Central|            5|
| Irbid|Northern|            5|
| Aqaba|Southern|          

### 9.5 Subqueries

In [75]:
# Apartments priced above the overall average
spark.sql("""
    SELECT City, Bedrooms, Price
    FROM apartments
    WHERE Price > (
        SELECT AVG(Price) FROM apartments
    )
    ORDER BY Price DESC
""").show()

# Cities whose average price exceeds the cross-city average
spark.sql("""
    SELECT City, ROUND(avg_price, 0) AS avg_price
    FROM (
        SELECT City, AVG(Price) AS avg_price
        FROM apartments
        GROUP BY City
    ) city_avg
    WHERE avg_price > (
        SELECT AVG(Price) FROM apartments
    )
    ORDER BY avg_price DESC
""").show()

+-----+--------+--------+
| City|Bedrooms|   Price|
+-----+--------+--------+
|Amman|       6|320000.0|
|Amman|       5|250000.0|
|Irbid|       5|190000.0|
|Amman|       4|185000.0|
|Irbid|       4|160000.0|
|Amman|       4|155000.0|
|Aqaba|       4|145000.0|
|Amman|       3|142000.0|
|Amman|       3|128000.0|
|Amman|       3|120000.0|
+-----+--------+--------+

+-----+---------+
| City|avg_price|
+-----+---------+
|Amman| 150200.0|
+-----+---------+



### 9.6 Window Functions

Window functions perform calculations over a set of rows related to the current row — without collapsing them into a single group.

In [76]:
# ROW_NUMBER, RANK within each city by price
spark.sql("""
    SELECT
        City,
        Bedrooms,
        Price,
        ROW_NUMBER() OVER (PARTITION BY City ORDER BY Price DESC) AS row_num,
        RANK()       OVER (PARTITION BY City ORDER BY Price DESC) AS price_rank,
        DENSE_RANK() OVER (PARTITION BY City ORDER BY Price DESC) AS dense_rank
    FROM apartments
    ORDER BY City, price_rank
""").show(15)

+-----+--------+--------+-------+----------+----------+
| City|Bedrooms|   Price|row_num|price_rank|dense_rank|
+-----+--------+--------+-------+----------+----------+
|Amman|       6|320000.0|      1|         1|         1|
|Amman|       5|250000.0|      2|         2|         2|
|Amman|       4|185000.0|      3|         3|         3|
|Amman|       4|155000.0|      4|         4|         4|
|Amman|       3|142000.0|      5|         5|         5|
|Amman|       3|128000.0|      6|         6|         6|
|Amman|       3|120000.0|      7|         7|         7|
|Amman|       2| 90000.0|      8|         8|         8|
|Amman|       2| 62000.0|      9|         9|         9|
|Amman|       1| 50000.0|     10|        10|        10|
|Aqaba|       4|145000.0|      1|         1|         1|
|Aqaba|       3|115000.0|      2|         2|         2|
|Aqaba|       3|108000.0|      3|         3|         3|
|Aqaba|       2| 88000.0|      4|         4|         4|
|Aqaba|       1| 38000.0|      5|         5|    

In [77]:
# LAG: compare each apartment's price to the previous listing in the same city
spark.sql("""
    SELECT
        City,
        Bedrooms,
        Price,
        LAG(Price, 1) OVER (PARTITION BY City ORDER BY Price)              AS prev_price,
        ROUND(Price - LAG(Price, 1) OVER (PARTITION BY City ORDER BY Price), 0) AS price_diff
    FROM apartments
    ORDER BY City, Price
""").show(15)

+-----+--------+--------+----------+----------+
| City|Bedrooms|   Price|prev_price|price_diff|
+-----+--------+--------+----------+----------+
|Amman|       1| 50000.0|      NULL|      NULL|
|Amman|       2| 62000.0|   50000.0|   12000.0|
|Amman|       2| 90000.0|   62000.0|   28000.0|
|Amman|       3|120000.0|   90000.0|   30000.0|
|Amman|       3|128000.0|  120000.0|    8000.0|
|Amman|       3|142000.0|  128000.0|   14000.0|
|Amman|       4|155000.0|  142000.0|   13000.0|
|Amman|       4|185000.0|  155000.0|   30000.0|
|Amman|       5|250000.0|  185000.0|   65000.0|
|Amman|       6|320000.0|  250000.0|   70000.0|
|Aqaba|       1| 38000.0|      NULL|      NULL|
|Aqaba|       2| 88000.0|   38000.0|   50000.0|
|Aqaba|       3|108000.0|   88000.0|   20000.0|
|Aqaba|       3|115000.0|  108000.0|    7000.0|
|Aqaba|       4|145000.0|  115000.0|   30000.0|
+-----+--------+--------+----------+----------+
only showing top 15 rows


In [78]:
# Window functions using the DataFrame API (equivalent to the SQL above)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag

window_spec = Window.partitionBy("City").orderBy(desc("Price"))

df_window = df.withColumn("row_num",    row_number().over(window_spec)) \
              .withColumn("price_rank", rank().over(window_spec)) \
              .withColumn("dense_rank", dense_rank().over(window_spec))

df_window.select("City", "Bedrooms", "Price", "row_num", "price_rank", "dense_rank") \
         .orderBy("City", "price_rank") \
         .show(15)

+-----+--------+--------+-------+----------+----------+
| City|Bedrooms|   Price|row_num|price_rank|dense_rank|
+-----+--------+--------+-------+----------+----------+
|Amman|       6|320000.0|      1|         1|         1|
|Amman|       5|250000.0|      2|         2|         2|
|Amman|       4|185000.0|      3|         3|         3|
|Amman|       4|155000.0|      4|         4|         4|
|Amman|       3|142000.0|      5|         5|         5|
|Amman|       3|128000.0|      6|         6|         6|
|Amman|       3|120000.0|      7|         7|         7|
|Amman|       2| 90000.0|      8|         8|         8|
|Amman|       2| 62000.0|      9|         9|         9|
|Amman|       1| 50000.0|     10|        10|        10|
|Aqaba|       4|145000.0|      1|         1|         1|
|Aqaba|       3|115000.0|      2|         2|         2|
|Aqaba|       3|108000.0|      3|         3|         3|
|Aqaba|       2| 88000.0|      4|         4|         4|
|Aqaba|       1| 38000.0|      5|         5|    

**Window function reference:**

| Function | Description |
|---|---|
| `ROW_NUMBER()` | Sequential integer, unique per row |
| `RANK()` | Rank with gaps (ties get same rank, next rank skips) |
| `DENSE_RANK()` | Rank without gaps |
| `LAG(col, n)` | Value n rows before the current row |
| `LEAD(col, n)` | Value n rows after the current row |
| `SUM() OVER (...)` | Running or windowed sum |
| `AVG() OVER (...)` | Running or windowed average |

<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  10. Reading &amp; Writing Data
</h2>

Spark can read from and write to many formats. The three most common are CSV, JSON, and Parquet.

### 10.1 CSV

In [79]:
csv_out = "/tmp/apartments_out.csv"

# Write CSV
df.write \
  .option("header", True) \
  .option("sep", ",") \
  .mode("overwrite") \
  .csv(csv_out)

print(f"CSV written to: {csv_out}")
import os
print("Output files:", os.listdir(csv_out))

# Read it back
df_read = spark.read.csv(csv_out, header=True, inferSchema=True)
print(f"\nRows read back: {df_read.count()}")
df_read.show(5)

CSV written to: /tmp/apartments_out.csv
Output files: ['.part-00001-4d6ce6cf-dd19-477b-9cb0-336b6bb887c8-c000.csv.crc', '._SUCCESS.crc', 'part-00001-4d6ce6cf-dd19-477b-9cb0-336b6bb887c8-c000.csv', '_SUCCESS', 'part-00000-4d6ce6cf-dd19-477b-9cb0-336b6bb887c8-c000.csv', '.part-00000-4d6ce6cf-dd19-477b-9cb0-336b6bb887c8-c000.csv.crc']

Rows read back: 25
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Irbid|       3|        2|      145.0| 98000.0|
|Amman|       2|        1|       80.0| 62000.0|
|Aqaba|       4|        3|      180.0|145000.0|
|Amman|       6|        5|      380.0|320000.0|
|Zarqa|       1|        1|       55.0| 40000.0|
+-----+--------+---------+-----------+--------+
only showing top 5 rows


> Spark writes one file **per partition** inside the output directory, plus `_SUCCESS` and `_committed_*` metadata files. This is normal distributed behaviour — all part-files together constitute the full dataset.

### 10.2 JSON

In [80]:
json_out = "/tmp/apartments_out.json"

# Write JSON lines
df.write \
  .mode("overwrite") \
  .json(json_out)

# Read it back
df_json_read = spark.read.json(json_out)
print(f"Rows read back: {df_json_read.count()}")
df_json_read.show(5)

Rows read back: 25
+---------+--------+-----+--------+-----------+
|Bathrooms|Bedrooms| City|   Price|Square_Area|
+---------+--------+-----+--------+-----------+
|        2|       3|Irbid| 98000.0|      145.0|
|        1|       2|Amman| 62000.0|       80.0|
|        3|       4|Aqaba|145000.0|      180.0|
|        5|       6|Amman|320000.0|      380.0|
|        1|       1|Zarqa| 40000.0|       55.0|
+---------+--------+-----+--------+-----------+
only showing top 5 rows


### 10.3 Parquet — The Recommended Format

**Why Parquet?**

| Feature | CSV | JSON | Parquet |
|---|---|---|---|
| Storage format | Row-based text | Row-based text | **Columnar binary** |
| Schema embedded | No (must infer) | Partial | **Yes** |
| Compression | None by default | None by default | **Snappy by default** |
| Read performance | Slow (full scan) | Slow (full scan) | **Fast (column pruning)** |
| Write performance | Fast | Medium | Medium |
| Predicate pushdown | No | No | **Yes** |
| Nested data types | No | Yes | **Yes** |

Parquet's columnar storage means Spark only reads the columns your query needs, dramatically reducing I/O.

In [81]:
parquet_out = "/tmp/apartments.parquet"

# Write Parquet (Snappy compression is default)
df.write \
  .mode("overwrite") \
  .parquet(parquet_out)

# Read Parquet — schema is preserved automatically
df_parquet = spark.read.parquet(parquet_out)
print("Schema preserved from Parquet file:")
df_parquet.printSchema()
df_parquet.show(5)

Schema preserved from Parquet file:
root
 |-- City: string (nullable = true)
 |-- Bedrooms: long (nullable = true)
 |-- Bathrooms: long (nullable = true)
 |-- Square_Area: double (nullable = true)
 |-- Price: double (nullable = true)

+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Irbid|       3|        2|      145.0| 98000.0|
|Amman|       2|        1|       80.0| 62000.0|
|Aqaba|       4|        3|      180.0|145000.0|
|Amman|       6|        5|      380.0|320000.0|
|Zarqa|       1|        1|       55.0| 40000.0|
+-----+--------+---------+-----------+--------+
only showing top 5 rows


### 10.4 Partitioned Writes

Partitioning splits the output into subdirectories by the value of one or more columns. This enables **partition pruning** — queries that filter on the partition column only read the relevant subdirectory, skipping all others.

In [83]:
partitioned_out = "/tmp/apartments_partitioned"

# Write partitioned by City
df.write \
  .partitionBy("City") \
  .mode("overwrite") \
  .parquet(partitioned_out)

# Show the directory structure
import os
for item in sorted(os.listdir(partitioned_out)):
    sub = os.path.join(partitioned_out, item)
    if os.path.isdir(sub) and not item.startswith("_"):
        print(f"  {partitioned_out}/{item}/")
        for f in os.listdir(sub):
            if not f.startswith("_"):
                print(f"    {f}")

  /tmp/apartments_partitioned/City=Amman/
    .part-00001-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet.crc
    part-00001-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet
    part-00000-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet
    .part-00000-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet.crc
  /tmp/apartments_partitioned/City=Aqaba/
    .part-00001-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet.crc
    part-00001-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet
    part-00000-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet
    .part-00000-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet.crc
  /tmp/apartments_partitioned/City=Irbid/
    .part-00001-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet.crc
    part-00001-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet
    part-00000-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000.snappy.parquet
    .part-00000-0eacfbbd-c09b-4acb-85da-3ac982fa5114.c000

In [84]:
# Spark automatically recognises partition columns when reading
df_part = spark.read.parquet(partitioned_out)
df_part.printSchema()

# Partition pruning: only the City=Amman partition is read from disk
df_part.filter(col("City") == "Amman").show()

root
 |-- Bedrooms: long (nullable = true)
 |-- Bathrooms: long (nullable = true)
 |-- Square_Area: double (nullable = true)
 |-- Price: double (nullable = true)
 |-- City: string (nullable = true)

+--------+---------+-----------+--------+-----+
|Bedrooms|Bathrooms|Square_Area|   Price| City|
+--------+---------+-----------+--------+-----+
|       3|        2|      150.0|120000.0|Amman|
|       1|        1|       60.0| 50000.0|Amman|
|       5|        4|      300.0|250000.0|Amman|
|       4|        3|      220.0|185000.0|Amman|
|       3|        2|      155.0|128000.0|Amman|
|       2|        1|       80.0| 62000.0|Amman|
|       6|        5|      380.0|320000.0|Amman|
|       3|        3|      170.0|142000.0|Amman|
|       4|        2|      190.0|155000.0|Amman|
|       2|        2|      110.0| 90000.0|Amman|
+--------+---------+-----------+--------+-----+



<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  11. Exercises
</h2>

Apply what you have learned to solve the following exercises. Use the `apartments` dataset (the `df` variable or the `apartments` SQL view).

### Exercise 1 — Affordable Family Apartments

**Task:** Find all apartments in Amman or Irbid that have at least 3 bedrooms and cost less than 130,000. Display the result ordered by price (cheapest first), showing only the columns: `City`, `Bedrooms`, `Square_Area`, `Price`.

Solve this **two ways**: once using the DataFrame API and once using Spark SQL.

In [85]:
# --- Exercise 1: DataFrame API Solution ---
ex1_df = (
    df
    .filter(
        (col("City").isin("Amman", "Irbid")) &
        (col("Bedrooms") >= 3) &
        (col("Price") < 130000)
    )
    .select("City", "Bedrooms", "Square_Area", "Price")
    .orderBy(asc("Price"))
)

print("DataFrame API result:")
ex1_df.show()

# --- Exercise 1: Spark SQL Solution ---
ex1_sql = spark.sql("""
    SELECT City, Bedrooms, Square_Area, Price
    FROM apartments
    WHERE City IN ('Amman', 'Irbid')
      AND Bedrooms >= 3
      AND Price < 130000
    ORDER BY Price ASC
""")

print("Spark SQL result:")
ex1_sql.show()

DataFrame API result:
+-----+--------+-----------+--------+
| City|Bedrooms|Square_Area|   Price|
+-----+--------+-----------+--------+
|Irbid|       3|      145.0| 98000.0|
|Amman|       3|      150.0|120000.0|
|Amman|       3|      155.0|128000.0|
+-----+--------+-----------+--------+

Spark SQL result:
+-----+--------+-----------+--------+
| City|Bedrooms|Square_Area|   Price|
+-----+--------+-----------+--------+
|Irbid|       3|      145.0| 98000.0|
|Amman|       3|      150.0|120000.0|
|Amman|       3|      155.0|128000.0|
+-----+--------+-----------+--------+



### Exercise 2 — City Price Report with Efficiency Score

**Task:** For each city, calculate:
- Total number of listings
- Average price (rounded to 0 decimal places)
- Average price per square metre (rounded to 2 decimal places)
- A column `Value_Score` = `'High'` if avg price per sqm < 800, `'Medium'` if < 1000, else `'Low'`

Order the result by average price per sqm ascending.

In [86]:
# --- Exercise 2 Solution ---
ex2 = (
    df
    .groupBy("City")
    .agg(
        count("*").alias("listings"),
        spark_round(avg("Price"), 0).alias("avg_price"),
        spark_round(avg(col("Price") / col("Square_Area")), 2).alias("avg_price_per_sqm")
    )
    .withColumn(
        "Value_Score",
        when(col("avg_price_per_sqm") < 800, "High")
        .when(col("avg_price_per_sqm") < 1000, "Medium")
        .otherwise("Low")
    )
    .orderBy(asc("avg_price_per_sqm"))
)

ex2.show()

+-----+--------+---------+-----------------+-----------+
| City|listings|avg_price|avg_price_per_sqm|Value_Score|
+-----+--------+---------+-----------------+-----------+
|Zarqa|       5|  78400.0|           715.08|       High|
|Irbid|       5| 117000.0|           751.69|       High|
|Aqaba|       5|  98800.0|            813.4|     Medium|
|Amman|      10| 150200.0|           821.98|     Medium|
+-----+--------+---------+-----------------+-----------+



### Exercise 3 — Top Listing per City Using Window Functions

**Task:** Using window functions, find the **single most expensive apartment** in each city. Display: `City`, `Bedrooms`, `Bathrooms`, `Square_Area`, `Price`. (Hint: Use `ROW_NUMBER()` partitioned by `City` and ordered by `Price DESC`, then filter `row_num = 1`.)

In [87]:
# --- Exercise 3 Solution ---
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

win = Window.partitionBy("City").orderBy(desc("Price"))

ex3 = (
    df
    .withColumn("row_num", row_number().over(win))
    .filter(col("row_num") == 1)
    .select("City", "Bedrooms", "Bathrooms", "Square_Area", "Price")
    .orderBy(desc("Price"))
)

print("Most expensive apartment per city:")
ex3.show()

# Equivalent Spark SQL solution
ex3_sql = spark.sql("""
    SELECT City, Bedrooms, Bathrooms, Square_Area, Price
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY City ORDER BY Price DESC) AS rn
        FROM apartments
    )
    WHERE rn = 1
    ORDER BY Price DESC
""")

print("Spark SQL result:")
ex3_sql.show()

Most expensive apartment per city:
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       6|        5|      380.0|320000.0|
|Irbid|       5|        3|      250.0|190000.0|
|Aqaba|       4|        3|      180.0|145000.0|
|Zarqa|       4|        2|      160.0|105000.0|
+-----+--------+---------+-----------+--------+

Spark SQL result:
+-----+--------+---------+-----------+--------+
| City|Bedrooms|Bathrooms|Square_Area|   Price|
+-----+--------+---------+-----------+--------+
|Amman|       6|        5|      380.0|320000.0|
|Irbid|       5|        3|      250.0|190000.0|
|Aqaba|       4|        3|      180.0|145000.0|
|Zarqa|       4|        2|      160.0|105000.0|
+-----+--------+---------+-----------+--------+



<h2 style="color:#E25822; font-family:Arial,sans-serif; border-bottom:2px solid #E25822; padding-bottom:4px;">
  Summary
</h2>

This notebook covered the complete lifecycle of structured data processing in PySpark:

| Section | Key takeaways |
|---|---|
| DataFrames vs RDDs | DataFrames are faster and easier for structured data due to Catalyst optimizer |
| Creating DataFrames | Row objects, typed schemas, CSV, JSON |
| Core Operations | select, filter, withColumn, drop, rename, sort, limit |
| Aggregations | groupBy + agg with count, avg, min, max |
| Joins | inner, left, right — same semantics as SQL |
| Built-in Functions | String, numeric, date, conditional — always prefer over UDFs |
| Missing Data | dropna, fillna, null counting |
| Spark SQL | Full SQL on DataFrames via temp views, including window functions |
| File Formats | Parquet preferred for analytics; CSV/JSON for interoperability |
| Partitioning | Critical for performance on large datasets |

### Next steps
- **Module 6:** Introduction to Databricks — managed Spark platform with Delta Lake
- Explore `spark.sql.functions` documentation for the full list of 300+ built-in functions
- Practice optimisation techniques: broadcast joins, caching (`df.cache()`), repartitioning

---
<p style="text-align:center; color:#888; font-size:0.9em;">Big Data Engineering &mdash; Module 5 &mdash; DataFrames and Spark SQL</p>

In [88]:
# Stop the SparkSession when finished
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
